# Lab - Accelerator Invocation
<div style="text-align: justify;">
This lab explores the fundamentals of invoking hardware accelerators using the DOCA Core library and the DOCA Execution Model on the NVIDIA BlueField DPU. Building on prior labs covering task submission, the Progress Engine, and memory management with <i>doca_mmap</i>, <i>doca_buf</i>, and <i>doca_buf_inventory</i>, this exercise focuses on configuring and utilizing the DOCA Compression Accelerator through the <i>doca_compress</i> API. The objective is to demonstrate how to create and submit compression tasks asynchronously and implement completion and error callbacks to handle task results. The lab provides hands-on experience with accelerator invocation via the DOCA Execution Model. A working knowledge of C programming is a recommended prerequisite.

</div>

# Introduction 

<div style="text-align: justify;">
The DOCA Execution Model (EM) enables BlueField DPU applications to submit work to hardware accelerators as asynchronous tasks. The Compression Accelerator (CA) part of the hardware accelerator provides hardware offload for data compression, reducing CPU usage and improving throughput for data-intensive workloads. The accelerator is accessed through the DOCA Core and DOCA Compress APIs [<a href="#References">1</a>].
</div> <br>

<div style="text-align: justify;">
Building on previous concepts such as Progress Engine (PE), memory mapping, and task submission, this lab introduces the Compression Context (CC) and the process of submitting compression tasks. The flow includes configuring the CC, preparing memory buffers, submitting tasks to the PE, and handling task completion or errors through callbacks. Understanding this workflow is needed when developing BlueField DPU applications that leverage hardware acceleration.
</div> <br>

<div style="text-align: justify;">
The DOCA EM can be illustrated using Local DMA Memcpy tasks as an example. Each task represents a memory copy operation that is submitted to the runtime. The PE progresses these tasks asynchronously, and the application is notified of their completion via callbacks.
</div> <br>

<div style="text-align: justify;">
The typical workflow begins with initializing the application state and configuring the necessary contexts, such as the DMA Context. After preparing memory and buffer descriptors, the application wraps source and destination memory regions into <i>doca_buf</i> structures and creates DMA Memcpy task descriptors. Once the tasks are ready, they are submitted using <i>doca_task_submit()</i>, and their progress is driven by the PE.
</div> <br>

<figure style="text-align: center;">
  <img src="./figs/compress.jpg" width="1400" style="display: block; margin: 0 auto;">
  <figcaption>Figure 1. Data Compression Flow using DOCA Compress Library.</figcaption>
</figure>

<div style="text-align: justify;">
In the figure, the compression operation begins with a source buffer (<i>src_buf</i>) containing the original uncompressed data. This buffer is passed into the DOCA Compression Library using a compression task. The task encapsulates all necessary metadata, such as buffer location, size, and user-defined checksum, and is submitted asynchronously through the PE.
</div> <br>

<div style="text-align: justify;">
Once the task is processed, the library produces a destination buffer (<i>dst_buf</i>) that holds the compressed data. Additionally, a checksum of the original <i>src_buf</i> is computed. This checksum is later used for validating the integrity of the decompressed data. The compressed buffer now contains a smaller footprint of the original data, optimizing memory usage and transmission efficiency.
</div> <br>


### Setting up the compression context

<div style="text-align: justify;">
To utilize the CA, the application must first initialize and configure a CC. This context encapsulates the parameters and state required to perform hardware-accelerated compression. The setup begins with creating the context using <i>doca_compress_create()</i>. The application then configures it by populating a <i>doca_compress_config</i> structure with properties such as the desired algorithm (e.g., LZ4 or Zlib), and sets the operation direction to compression. Once configured, the context is associated with a PE by registering callback functions using <i>doca_ctx_set_task_completion_cb()</i> and starting the context with <i>doca_ctx_start()</i>. This binding ensures that the PE is responsible for driving task completion, enabling asynchronous execution and event-driven processing.
</div> <br>

### Preparing buffers and memory mapping

<div style="text-align: justify;">
With the context initialized, memory resources must be allocated and mapped for DPU access. This involves defining memory regions through <i>doca_mmap_add_mem_range()</i> and finalizing the memory map using <i>doca_mmap_start()</i>. The <i>doca_buf_inventory</i> is used to manage buffer descriptors, which are lightweight handles describing memory blocks. Each input and output region is wrapped with a <i>doca_buf</i>, enabling the accelerator to access data directly without additional CPU-side copies. This zero-copy model improves efficiency and is essential for hardware-accelerated workflows.
</div> <br>

### Submitting compression jobs

<div style="text-align: justify;">
Job submission is carried out through a <i>doca_compress_job</i> structure, initialized using <i>doca_compress_job_init()</i>. This job references the configured source and destination <i>doca_bufs</i>, and includes fields like input and output lengths. Once initialized, the job is submitted using <i>doca_ctx_submit()</i> to the CC managed by the PE. As the compression task is processed asynchronously, the application remains free to perform other tasks or submit additional jobs.
</div> <br>

<div style="text-align: justify;">
After completion, the application receives a notification through the callback mechanism registered earlier. The compressed data resides in the destination buffer, and a checksum of the original input may also be generated for future validation during decompression.
</div> <br>

### Handling task completion and errors

<div style="text-align: justify;">
As jobs progress, the application is notified via the registered callbacks. On success, the completion callback retrieves the compressed data and validates or stores the result. In the event of failure, the error callback uses <i>doca_task_get_status()</i> and <i>doca_error_get_descr()</i> to interpret the error and log diagnostics. Both paths involve cleaning up buffer references and freeing the job structure to ensure efficient resource usage and prevent memory leaks.
</div> <br>

### Cleaning up resources

<div style="text-align: justify;">
After processing is complete, all components of the pipeline are torn down in reverse order of initialization. <i>doca_compress_destroy()</i> is called to release the CC. Buffer inventories, memory mappings, and descriptors are destroyed, followed by shutting down the PE. This systematic cleanup ensures the application exits gracefully and leaves the system in a reusable state for future runs or labs.
</div> <br>

### Understanding the decompression flow

<figure style="text-align: center;">
  <img src="./figs/decompress.jpg" width="1400" style="display: block; margin: 0 auto;">
  <figcaption>Figure 2. Data Compression Flow using DOCA Compress Library.</figcaption>
</figure>

<div style="text-align: justify;">
In the decompression flow, decompression reverses the compression process. The source buffer (src_buf) contains previously compressed data and the checksum of the original (uncompressed) data. This src_buf is submitted to the DOCA Compression Library through a decompression task, again using the PE to handle asynchronous execution.
</div> <br>

<div style="text-align: justify;">
The decompressed output is written to the destination buffer (dst_buf), which should match the original uncompressed content (as it existed before compression). After decompression, a checksum is calculated on the dst_buf. This is compared with the checksum from the original compression step to verify data integrity.
</div> <br>

### Step 1:  Configuring the environment

<div style="text-align: justify;">
Before running this notebook, you will need to configure your environment using the <a href="../../configure_and_validate.ipynb" target="_blank">Configure Environment</a> notebook. Please stop here, open and run that notebook, then return to this notebook.
</div> <br>

<div style="text-align: justify;">
If you are using the FABRIC JupyterHub many of the environment variables will be automatically configured for you.  You will still need to set your bastion username, upload your bastion private key, and set the path to where you put your bastion private key. Your bastion username and private key should already be in your possession.  
</div> <br>

<div style="text-align: justify;">
If you are using the FABRIC API outside of the JupyterHub you will need to configure all of the environment variables. Defaults below will be correct in many situations, but you will need to confirm your configuration.  If you have questions about this configuration, please contact the FABRIC admins using the <a href="https://learn.fabric-testbed.net/forums/" target="_blank">FABRIC User Forum</a>
</div> <br>

<div style="text-align: justify;">
More information about accessing your experiments through the FABRIC bastion hosts can be found <a href="https://learn.fabric-testbed.net/knowledge-base/logging-into-fabric-vms/" target="_blank">here</a>.
</div>

### Step 2: Importing the FABlib library

In [7]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
fablib = fablib_manager()

User: Amithgspn@sc.edu bastion key is valid!
Configuration is valid


### Step 3: Creating the experiment slice

<div style="text-align: justify;">
This example sets up two nodes, each equipped with a BlueField NIC, connected to an isolated local Ethernet. 
</div> <br>

<div style="text-align: justify;">
Each node is created with a single NIC component, utilizing the <code>NIC_ConnectX_7_400</code> and <code>NIC_Basic</code> models. These components are attached to the node via PCI passthrough. A list of other available NIC models is provided below. To retrieve the interfaces associated with a NIC component, use the <code>get_interfaces()</code> method. Many dedicated NICs feature multiple ports, either of which can be connected to the network.
</div> <br>

<div style="text-align: justify;">
The node connected to the BlueField SmartNIC runs the <code>default_ubuntu_24</code> image and manually install a DOCA version of <code>3.0.0</code>. 
</div> <br>

<div style="text-align: justify;">
For automatic configuration, specify a subnet for the network and set the interface mode to <code>auto</code> using <code>iface1.set_mode('auto')</code> before submitting the request. With this setup, FABlib assigns an IP address from the subnet and configures the device during post-boot setup. Additionally, routes can be pre-configured before submitting the request.
</div> 

#### Available NIC Component Models:
- **NIC_Basic**: 100 Gbps Mellanox ConnectX-6 SR-IOV VF (1 Port)
- **NIC_ConnectX_5**: 25 Gbps Dedicated Mellanox ConnectX-5 PCI Device (2 Ports)
- **NIC_ConnectX_6**: 100 Gbps Dedicated Mellanox ConnectX-6 PCI Device (2 Ports)
- **NIC_ConnectX_7_100**: 100 Gbps Dedicated Mellanox BlueField-3 ConnectX-7 PCI Device (2 Ports)
- **NIC_ConnectX_7_400**: 400 Gbps Dedicated Mellanox BlueField-3 ConnectX-7 PCI Device (2 Ports)
- **NIC_BlueField2_ConnectX_6**: 100 Gbps Dedicated Mellanox BlueField-2 ConnectX-6 PCI Device (2 Ports)

#### Step 3.1: Creating a slice

The available sites for 100 Gbps DPUs are: 
- MASS, DALL, FIU, LOSA, NEWY, ATLA, SEAT, and HAWI.

The available sites for 400 Gbps DPUs are:
- SALT and WASH.  

<div style="text-align: justify;">
In this experiment, we selected the MASS site for deployment.  
For other experiments, replace FIU with the desired site name from the list above.
</div>

In [8]:
slice_name = 'DOCA_LAB'

site="FIU" 
print(f"Site: {site}")

node1_name = 'Node1'

node1_image = "default_ubuntu_24"

slice = fablib.new_slice(name=slice_name)

Site: FIU


#### Step 3.2: Submitting the slice

<div style="text-align: justify;">
The code below submits the slice. 
By default, the submit function will block until the node is ready and will display the progress of your slice being built.
</div>

In [ ]:
# Node1
node1 = slice.add_node(name=node1_name, site=site, image=node1_image, disk=30)
dpu = node1.add_component(model='NIC_ConnectX_7_100', name='nic1')
iface1 =  dpu.get_interfaces()[0]
iface1.set_mode('auto')

iface2 =  dpu.get_interfaces()[1]
iface2.set_mode('auto')

# Submit Slice Request
slice.submit();

<div style="text-align: justify;">
The commands below retrieve the specified FABRIC slice and display its components. They list all nodes, networks, and interfaces within the slice, providing an overview of the allocated resources and their connectivity. </div>

In [10]:
slice=fablib.get_slice(slice_name)
slice.list_nodes()
slice.list_networks();
slice.list_interfaces();

ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
93751853-a8c4-4485-b092-b1de1150ccf5,Node1,2,8,100,default_ubuntu_24,qcow2,fiu-w3.fabric-testbed.net,FIU,ubuntu,131.94.57.46,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@131.94.57.46,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1-nic1-p1,p1,Node1,None,100,auto,,CC:40:F3:80:01:FC,enp7s0np0,enp7s0np0,None,7,None
Node1-nic1-p2,p2,Node1,None,100,auto,,CC:40:F3:80:01:FD,enp8s0np1,enp8s0np1,None,7,None


#### Step 3.3 Getting the server node
<div style="text-align: justify;">
The command below gets the fablib node that Node1 is associated with.
</div>

In [11]:
node1 = slice.get_node(name="Node1")

#### Step 3.4 Installing dependencies
<div style="text-align: justify;">
The code below installs packages that are prerequisites to the upcoming installations and needed to run the lab experiments
</div>

In [34]:
stdout, stderr = node1.execute(f'sudo apt-get update', quiet = True)
stdout, stderr = node1.execute(f'sudo apt-get install -y docker.io build-essential python3-pip net-tools', quiet = True)
stdout, stderr = node1.execute(f'sudo pip3 install meson ninja', quiet = True)
stdout, stderr = node1.execute(
    "ssh ubuntu@192.168.100.2 'sudo apt-get update && sudo apt-get -y install meson ninja-build'"
)

Get:1 file:/var/hbn-repo-aarch64-ubuntu2204-local ./ InRelease
Ign:1 file:/var/hbn-repo-aarch64-ubuntu2204-local ./ InRelease
Get:2 file:/var/hbn-repo-aarch64-ubuntu2204-local ./ Release [466 B]
Get:2 file:/var/hbn-repo-aarch64-ubuntu2204-local ./ Release [466 B]
Get:3 file:/var/hbn-repo-aarch64-ubuntu2204-local ./ Release.gpg
Ign:3 file:/var/hbn-repo-aarch64-ubuntu2204-local ./ Release.gpg
Get:4 file:/var/hbn-repo-aarch64-ubuntu2204-local ./ Packages [1470 B]
Get:6 http://ports.ubuntu.com/ubuntu-ports jammy InRelease [270 kB]
Get:5 https://prod-cdn.packages.k8s.io/repositories/isv:/kubernetes:/core:/stable:/v1.30/deb  InRelease [1192 B]
Ign:7 https://linux.mellanox.com/public/repo/doca/latest/ubuntu22.04/aarch64 ./ InRelease
Get:8 https://prod-cdn.packages.k8s.io/repositories/isv:/kubernetes:/core:/stable:/v1.30/deb  Packages [20.3 kB]
Get:9 http://ports.ubuntu.com/ubuntu-ports jammy-updates InRelease [128 kB]
Get:10 http://ports.ubuntu.com/ubuntu-ports jammy-backports InRelease [127 

KeyboardInterrupt: 

#### Step 3.5: Installing NVIDIA DOCA 3.0.0 software on the host

<div style="text-align: justify;">
The code below installs packages that are prerequisites to the upcoming installations and are needed to run the lab experiments
</div>

In [21]:
stdout, stderr = node1.execute(f'wget https://www.mellanox.com/downloads/DOCA/DOCA_v3.0.0_VG/host/doca-host_3.0.0-058001-25.04-ubuntu2404_amd64.deb', quiet = True)

In [22]:
stdout, stderr = node1.execute(f'sudo dpkg -i doca-host_3.0.0-058001-25.04-ubuntu2404_amd64.deb', quiet = True)

In [23]:
stdout, stderr = node1.execute(f'sudo apt-get update', quiet = True)

In [ ]:
stdout, stderr = node1.execute(f'sudo apt-get -y install doca-all', quiet = True)

In [26]:
stdout, stderr = node1.execute(f'sudo systemctl enable rshim', quiet = True)

stdout, stderr = node1.execute(f'sudo systemctl start rshim', quiet = True)

#### Step 3.6: Installing NVIDIA DOCA 3.0.0 software on the Bluefield Smart NIC

In [ ]:
stdout, stderr = node1.execute(f'wget https://content.mellanox.com/BlueField/BFBs/Ubuntu22.04/bf-bundle-3.0.0-135_25.04_ubuntu-22.04_prod.bfb', quiet = True)

In [28]:
stdout, stderr = node1.execute(f'bfb-install --bfb bf-bundle-3.0.0-135_25.04_ubuntu-22.04_prod.bfb --rshim rshim0')

Checking if local host has root access...
Checking if rshim driver is running locally...
Warn: 'pv' command not found. Continue without showing BFB progress.
Pushing bfb
cat: /dev/rshim0/misc: Permission denied
 INFO[PSC]: PSC BL1 START
 INFO[BL2]: start
 INFO[BL2]: boot mode (rshim)
 INFO[BL2]: VDD_CPU: 790 mV
 INFO[BL2]: VDDQ: 1120 mV
 INFO[BL2]: DDR POST passed
 INFO[BL2]: UEFI loaded
 INFO[BL31]: start
 INFO[BL31]: lifecycle GA Secured
 INFO[BL31]: runtime
 INFO[BL31]: MB ping success
 INFO[UEFI]: eMMC init
 INFO[UEFI]: eMMC probed
 INFO[UEFI]: UPVS valid
 INFO[UEFI]: PMI: updates started
 INFO[UEFI]: PMI: total updates: 1
 INFO[UEFI]: PMI: updates completed, status 0
 INFO[UEFI]: PCIe enum start
 INFO[UEFI]: PCIe enum end
 INFO[UEFI]: UEFI Secure Boot (enabled)
 INFO[UEFI]: Redfish enabled
 INFO[UEFI]: exit Boot Service
 INFO[MISC]: Erasing eMMC drive: /dev/mmcblk0
 INFO[MISC]: Erasing NVME drive: /dev/nvme0n1
 INFO[MISC]: Ubuntu installation started
 INFO[MISC]: Installing OS ima

#### Step 3.7: Rebooting VM in Node 1 after installing the software

<div style="text-align: justify;">
After installing the DOCA software, reboot the VM (Node 1) connected to the DPU, since the VM OS may not always detect the interfaces automatically.
</div>

In [29]:
stdout, stderr = node1.execute("sudo reboot")
slice.wait_ssh()

True

#### Step 3.8: Assigning an IP to the Bluefield Smart NIC

<div style="text-align: justify;">
The BlueField SmartNIC is configured by assigning the private IP address <code>192.168.100.1</code> to the <code>tmfifo_net0</code> device, enabling communication and management of the BlueField DPU. Additionally, the BlueField bundle (BFB) is installed on the DPU via the designated RShim interface, ensuring firmware updates and configuration optimizations for enhanced data center performance.
</div>

In [32]:
stdout, stderr = node1.execute("sudo ip addr add 192.168.100.1/24 dev tmfifo_net0")

Error: ipv4: Address already assigned.


In [33]:
stdout, stderr = node1.execute("sudo ip link set tmfifo_net0 up")

#### Step 3.9: Accessing DPU:
##### SSH Access:
To SSH into DPU from the VM, use:
```
ssh ubuntu@192.168.100.2
```
After pushing the BFB image to DPU, the default credentials are:  
- **Username:** `ubuntu`  
- **Password:** `ubuntu` (you will be prompted to change it upon first login)

##### Console Access:
If SSH access is unavailable, you can connect to DPU via the console interface:
```
screen /dev/rshim0/console
```
This method provides an alternative way to manage the SmartNIC if SSH connectivity is lost.

##### Bypass without password. Execute the following commands in host:
```
# 1. Generate key pair (skip if already exists)
ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519 -N ''

# 2. Copy the public key into BlueField
ssh-copy-id ubuntu@192.168.100.2
```

### Step 4: Writing and compiling the main program
This section demonstrates how to define the main program. 

<div style="text-align: justify;">
The files that are needed to compile, build, and run the C program for this application are defined as follows:
</div>

<ul style="text-align: justify;">
  <li>
    <strong><i>main.c:</i></strong> entry point of the program; sets up logging and invokes core logic.
  </li>

  <li>
    <strong><i>logic.c:</i></strong> implements the main execution flow including initialization, task setup, and job submission.
  </li>

  <li>
    <strong><i>definitions.h:</i></strong> contains global constants, enums, and shared data structures.
  </li>

  <li>
    <strong><i>meson.build:</i></strong> specifies the build rules, source files, and DOCA library dependencies.
  </li>

  <li>
    <strong><i>callbacks.c:</i></strong> defines completion and error callback functions for compression tasks.
  </li>

  <li>
    <strong><i>callbacks.h:</i></strong> declares the compression task callback function prototypes.
  </li>

  <li>
    <strong><i>utils.c:</i></strong> provides helper functions like file reading and error-checking macros.
  </li>

  <li>
    <strong><i>utils.h:</i></strong> declares utility functions and includes required DOCA and system headers.
  </li>

  <li>
    <strong><i>infile:</i></strong> binary input file used for testing compression and decompression.
  </li>

  <li>
    <strong><i>decompressed:</i></strong> output file generated after successful decompression for verification.
  </li>
</ul>

#### Step 4.1:	Defining the main program


<div style="text-align: justify;">
Click on the <i>main.c</i> file to display the content of the file. Use the file explorer on the left-hand side of the screen to locate the file.
</div>

<figure style="text-align: center;">
  <img src="./figs/main/fig1.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 3. Inspecting the <i>main.c</i> file</figcaption>
</figure>

<div style="text-align: justify;">
In the figure, the <i>main.c</i> file includes the standard C header <i>stdlib.h</i> for basic functions like memory management and process control. It also includes <i>doca_error.h</i> to handle errors returned by DOCA functions. The <i>doca_log.h</i> header is used to log messages during program execution. Line 8 registers a logging context named <i>LAB5::MAIN</i>, which tags all log messages from this source file. This helps identify where logs originate during debugging or analysis.
</div> <br>

<div style="text-align: justify;">
The return type <i>doca_error_t</i> is a custom type defined by the DOCA SDK, used to indicate the outcome of a function, typically success (<i>DOCA_SUCCESS</i>) or an error code (<i>DOCA_ERROR</i>). The <i>run_lab_logi</i>c function is invoked from main.c and is defined in logic.c using the same name. The (void) in the parameter list specifies that this function does not accept any arguments.
</div> <br>

**NOTE:** The <i>main.c</i> and <i>logic.c</i> files are linked to the <i>meson.build</i> file for compilation.
<hr>

<div style="text-align: justify;">
Next, we will be inspecting the program logic function.
</div> <br>

<figure style="text-align: center;">
  <img src="./figs/main/fig2.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 4. Defining the program logic function.</figcaption>
</figure>

In the figure, the definition of the main function serves as the program's entry point. It takes two arguments: <i>argc</i> (argument count) and <i>argv</i> (argument vector), enabling the program to handle command-line inputs. We are avoiding the compiler warnings of the unused variables by writing <i>(void)(argc)</i> and <i>(void)(argv)</i>.

**NOTE:** We will not be using <i>argc</i> and <i>argv</i> parameters in this lab, but we still include them since this is the standard way to write a C program. 
<hr>

<div style="text-align: justify;">
Now, we have the variables for logging and the program result predefined. These variables help monitor the program’s execution and determine if it completes successfully or with an error. </div> <br>

<figure style="text-align: center;">
  <img src="./figs/main/fig3.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 5. Defining the variables for logging and the program result.</figcaption>
</figure>

<div style="text-align: justify;">
In the above figure, the variable <i>doca_error_t/i> result is used to store the return status of DOCA API calls, allowing the program to verify whether each operation was successful. The <i>sdk_log</i> pointer, of type struct <i>doca_log_backend</i>, is defined in <i>doca_log.h</i> and is used to configure the logging behavior for the DOCA SDK, such as setting log output and levels. The <i>exit_status</i> variable is initialized to <i>EXIT_FAILURE</i> and is updated to <i>EXIT_SUCCESS</i> only if the program completes all tasks without error, ultimately serving as the return value from the main() function. </div> 
<hr>


<div style="text-align: justify;">
A standard logger is predefined in the program to enable the printing of general log messages, including information, warnings, and errors. </div> <br>

<figure style="text-align: center;">
  <img src="./figs/main/fig4.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 6. Registering standard logger backend.</figcaption>
</figure>

<div style="text-align: justify;">
In the figure, the <i>doca_log_backend_create_standard()</i> function initializes the standard logging system used by the application to display informational messages, warnings, and errors. This setup is essential for monitoring and debugging during execution. If the logger fails to initialize (i.e., the return value is not <i>DOCA_SUCCESS</i>), the program exits immediately using <i>EXIT_FAILURE</i>, a predefined constant (typically with the value 1) that signals abnormal program termination. </div> 
<hr>

<div style="text-align: justify;">
Now, the logging system is already initialized to display important messages from the DOCA SDK throughout the program's execution. </div> <br>

<figure style="text-align: center;">
  <img src="./figs/main/fig5.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 7. Defining the SDK logger and print starting message.</figcaption>
</figure>

<div style="text-align: justify;">
In the figure, the <i>doca_log_backend_create_with_file_sdk</i> API creates a logger that prints DOCA SDK warning and error messages to the screen using <i>stderr</i>. The function takes two parameters: <i>stderr</i> specifies the output stream for the log messages, and <i>&sdk_log</i> stores the created logger instance. This stored reference is later used to manage SDK log settings such as log level and output behavior. </div> <br>

<div style="text-align: justify;">
The <i>doca_log_backend_set_sdk_level</i> API is used to set the minimum severity level of log messages to be displayed. In this case, it is set to <i>DOCA_LOG_LEVEL_WARNING</i>, meaning only warning and error messages from DOCA SDK will appear. The <i>sdk_log</i> parameter is used here because it holds a reference to the logger instance created earlier; this allows the function to apply the log level setting to that specific logger. This approach helps reduce unnecessary output and keeps the logs focused on critical issues. If either of these steps fail, the program jumps to the <i>lab_exit</i> section to log the final status and return an appropriate exit code. </div> <br>

<div style="text-align: justify;">
The <i>DOCA_LOG_INFO</i> is a logging macro, a preprocessor directive provided by the DOCA SDK that simplifies logging during program execution. A macro is a reusable snippet of code defined to automate repetitive tasks. It is used to print informational messages during program execution. We use this macro to print a message “Starting the program LAB 5”, which can be seen in the terminal or log output. </div>

<hr>

<div style="text-align: justify;">
The main logic function is executed, and its return status is checked to confirm successful completion. </div> <br>

<figure style="text-align: center;">
  <img src="./figs/main/fig6.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 8. Running the lab logic function and handling errors.</figcaption>
</figure>

<div style="text-align: justify;">
In the figure, this code block runs the main logic of the lab and handles its result. The function <i>run_lab_logic()</i> is called, and its return value is stored in the result variable. This function contains the core logic of the program and is defined separately in <i>logic.c</i>. After the function call, the program checks if the result equals DOCA_SUCCESS, which indicates successful execution. If the function fails, an error message is logged using <i>DOCA_LOG_ERR</i>, and the program jumps to the <i>lab_exit</i> section to stop execution and handle cleanup. The error message includes a readable description of the error, retrieved using <i>doca_error_get_descr(result)</i>, which converts the error code into a meaningful string for easier debugging. If the function completes successfully, the program sets <i>exit_status</i> to <i>EXIT_SUCCESS</i>, which indicates a clean and successful run. </div>

<hr>

<div style="text-align: justify;">
Now we run the final section of the program to log the outcome and return the exit status. Add the following code to the <i>main.c</i> file. </div> <br>

<figure style="text-align: center;">
  <img src="./figs/main/fig7.png" width="1000" style="display: block; margin: 0 auto;">
  <figcaption>Figure 9. Logging the final program status and returning the exit code.</figcaption>
</figure>

<div style="text-align: justify;">
In the figure, the code checks whether <i>exit_status</i> is set to <i>EXIT_SUCCESS</i>. If so, it logs a message indicating that "LAB 5 finished successfully". Otherwise, it logs that the lab finished with errors. Finally, the program returns <i>exit_status</i> to indicate the result of the program execution. </div>

<hr>

#### Step 4.2:	Defining the definitions header

<div style="text-align: justify;">
The definitions.h file stores common definitions, such as structures, constants, and sometimes function declarations, which are shared across multiple source files (like main.c and logic.c in the lab5 directory). </div>

<div style="text-align: justify;">
Click on definitions.h in the file explorer panel on the left-hand side to open the header file in the editor. </div> <br>

<figure style="text-align: center;">
  <img src="./figs/definitions/fig1.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 10. Logging the final program status and returning the exit code.</figcaption>
</figure>

<hr>

<div style="text-align: justify;">
The definitions.h file has pre-defined header files. </div>

<figure style="text-align: center;">
  <img src="./figs/definitions/fig2.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 10. Logging the final program status and returning the exit code.</figcaption>
</figure>

<div style="text-align: justify;">
In this figure, we begin by checking if the macro <i>DEFINITIONS_H</i> is already defined. If it is not, we define it using <i>#define</i>. This ensures the header file is included only once during compilation to avoid duplicate definitions. The <i>#include <stdlib.h></i> directive provides functions for memory management and general utilities, while <i>#include <stdint.h></i> brings in definitions for fixed-width integer types, ensuring consistent behavior across different platforms. </div> <br>

<div style="text-align: justify;">
We end the conditional statement that we defined earlier with <i>#ifndef DEFINITIONS_H</i>. The <i>#endif</i> statement completes the inclusion check, ensuring that the contents of this header file are only included once during compilation.
</div> <br>

<div style="text-align: justify;">
The constants used to configure DMA task parameters and memory allocation are defined. The number of tasks (<i>NUM_TASKS</i>), size of each DMA buffer (<i>DMA_BUFFER_SIZE</i>), total buffer size (<i>BUFFER_SIZE</i>), and the size of the buffer inventory (<i>BUF_INVENTORY_SIZE</i>) are defined. These constants ensure that the application allocates sufficient memory and buffer descriptors to support multiple concurrent DMA Memcpy tasks. </div> 

<hr>

<div style="text-align: justify;">
We will now declare the <i>program_base</i> structure to store essential resources such as the DOCA device, buffer sizes, memory mappings, and buffer inventory. This structure centralizes all core elements required for program execution and memory management. Add the following code to <i>definitions.h file</i>. </div>

    struct program_base {
        struct doca_dev *device;
        struct doca_mmap *src_mmap;
        struct doca_mmap *dst_mmap;
        struct doca_buf_inventory *inventory;
        struct doca_pe *pe;
    
        size_t buffer_size;
        size_t buf_inventory_size;
    
        uint8_t *buffer;
        uint8_t *available_buffer; 
    
        uint32_t num_completed_tasks;
    };

<figure style="text-align: center;">
  <img src="./figs/definitions/fig3.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 11. Structure defining core program resources.</figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the <i>program_base</i> structure holds the key program resources required for task execution and memory management. It includes references to the DOCA device (<i>doca_dev</i>), memory map (<i>doca_mmap</i>), buffer inventory (<i>doca_buf_inventory</i>), and PE (<i>doca_pe</i>). It also stores buffer-related parameters such as buffer size, buffer pointers, and a counter to track the number of completed tasks.</div>  

<hr>

<div style="text-align: justify;">
We will now define a structure that stores runtime information required by the program, such as device handles or configuration states. This structure will be declared within the <i>definitions.h</i> file to make it accessible across all relevant source files. Add the following code to <i>definitions.h</i> file. </div>

    struct program_state {
        struct program_base base;
    
        struct doca_compress *compress;
        struct doca_ctx *compress_ctx;
        bool run_pe_progress;        
        size_t num_remaining_tasks;     
    };

<figure style="text-align: center;">
  <img src="./figs/definitions/fig4.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 12. Definition of program_state holding a pointer to the selected DOCA device.</figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the <i>program_state</i> structure is defined to hold the complete runtime state of the compression application. It embeds a <i>program_base</i> structure, which contains common resources such as buffers and device handles. Additionally, it introduces two compression-specific fields: a pointer to the DOCA Compression object (<i>compress</i>) and its associated context (<i>compress_ctx</i>). It also tracks the progress of the task processing loop using the <i>run_pe_progres</i> flag and maintains a counter for the number of remaining tasks via <i>num_remaining_tasks</i>. </div>  <br>

<div style="text-align: justify;">
<b>NOTE:</b> The files are linked together during the build process, allowing the structure to be referenced and used across source files.
<hr>

<div style="text-align: justify;">
We will now define the compression result structure. Add the following code to <i>definitions.h</i> file. </div>

    struct compress_deflate_result {
        doca_error_t status; 
        uint32_t crc_cs;     
        uint32_t adler_cs;   
    };

<figure style="text-align: center;">
  <img src="./figs/definitions/fig5.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 13. Structure to Store Compression Task Status and Checksums. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the <i>compress_deflate_result</i> structure is defined to hold the outcome of a compression task. It includes the status field of type <i>doca_error_t</i>, which indicates whether the compression operation succeeded or failed. Additionally, it stores two checksums: <i>crc_cs</i> and <i>adler_cs</i>, which are used to verify the integrity of the compressed data. These checksum values are later used to verify that the decompressed data matches the original, ensuring the compression worked correctly.</div>  <br>

<div style="text-align: justify;">
Save the changes and close the file.
</div>

<hr>

#### Step 4.3:	Defining the logic program

<div style="text-align: justify;">
Click on the <i>logic.c</i> file to display the contents of the file. Use the file explorer on the left-hand side of the screen to locate the  </div>

<figure style="text-align: center;">
  <img src="./figs/logic/fig1.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 14. Inspecting the logic file. </figcaption>
</figure>

<hr>

<div style="text-align: justify;">
The <i>logic.c</i> file contains predefined standard and DOCA header file inclusions and a logger for the lab 5 program logic function, as shown inside the gray box.</div>  <br>

<figure style="text-align: center;">
  <img src="./figs/logic/fig2.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 15. Inspecting the headers. </figcaption>
</figure>

<div style="text-align: justify;">
In the figure, the standard C library headers included are <i>stdio.h</i> for standard input and output operations, <i>stdlib.h</i> for memory management, process control, and utility functions, and <i>stdint.h</i> for fixed-width integer types ensuring portability. Additionally, several DOCA core utility headers are present: <i>doca_ctx.h</i> manages contexts required for task execution, doca_dev.h is essential for discovering and opening DOCA-compatible devices, <i>doca_dma.h</i> facilitates memory copy operations via DOCA’s DMA API, and <i>doca_types.h</i> introduces common data types and structures specific to DOCA. doca_buf.h provides functionality for buffer descriptor management, <i>doca_log.h</i> enables message logging, <i>doca_buf_inventory.h</i> supports efficient buffer management and reuse, <i>doca_mmap.h</i> handles memory mapping to enable memory sharing between host and DOCA devices, and <i>doca_pe.h</i> exposes APIs for controlling the PE that advances task execution. In addition, three user-defined header files are included: <i>definitions.h</i> defines shared data structures used across the program, <i>callbacks.h</i> declares callback functions for task completion and error handling, and <i>utils.h</i> provides utility macros and helper functions used throughout the application.
</div> <br>

<div style="text-align: justify;">
A logger namespace named <i>LAB5::LOGIC</i> is predefined to tag all log messages from this file, simplifying tracking and filtering during debugging.
</div>

<hr>

<div style="text-align: justify;">
We now initialize the program state. </div>  <br>

<figure style="text-align: center;">
  <img src="./figs/logic/fig3.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 16. Initializing the program state. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the <i>run</i> function begins by setting up the initial state required for the compression workflow. A <i>timespec</i> structure is initialized to configure a delay period in nanoseconds, which can be used for controlled task polling or throttling. Following this, the application logs the initialization step using <i>DOCA_LOG_INFO</i>, and then clears the memory associated with the <i>program_state</i> structure using <i>memset</i>. This ensures that all fields in the state structure are set to a known zeroed state before further configuration, helping prevent undefined behavior due to leftover data.
</div>

<hr>

<div style="text-align: justify;">
We will now be allocating the main application buffer. </div>  <br>

<figure style="text-align: center;">
  <img src="./figs/logic/fig4.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 17. Allocating buffer for compression workflow. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the program allocates a main buffer to store data for compression and decompression tasks. The size of the buffer is determined by the constant <i>BUFFER_SIZE</i>, and memory is dynamically allocated using <i>malloc</i>. This buffer is stored in the <i>base.buffer</i> field of the <i>program_state</i> structure. Additionally, the buffer inventory size is initialized, and a pointer to the available memory region is set for tracking usage during buffer assignment. If the memory allocation fails, the function immediately returns an out-of-memory error to prevent further execution with invalid memory references.
</div>

<hr>

<div style="text-align: justify;">
We will now be selecting a compatible DOCA device. </div>  <br>

<figure style="text-align: center;">
  <img src="./figs/logic/fig5.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 18. Opening a compatible DOCA device through device discovery. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application identifies and selects a DOCA-compatible device that supports deflate decompression tasks. It begins by generating a list of available DOCA devices using <i>doca_devinfo_create_list()</i>. If the list cannot be created, an error message is logged and the function returns early. The code then iterates through the devices, checking each one for decompression support using <i>doca_compress_cap_task_decompress_deflate_is_supported()</i>. Upon finding a suitable device, it attempts to open it with <i>doca_dev_open()</i> and stores the handle in the program state. Once a valid device is successfully opened, the device list is destroyed, and the loop exits, preparing the application to proceed with compression or decompression operations.
</div>

<hr>

<div style="text-align: justify;">
Now we will create and configure Memory Map (MMAP). </div>  <br>

<figure style="text-align: center;">
  <img src="./figs/logic/fig6.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 19. Creating and associating MMAP with the selected device. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application sets up memory mappings for both the source and destination buffers using the DOCA <i>mmap</i> API. It first creates two separate memory maps using the function <i>doca_mmap_create</i>, one for the source named <i>src_mmap</i> and one for the destination named <i>dst_mmap</i>. These memory maps are then associated with the selected DOCA device using <i>doca_mmap_add_dev</i>. 
</div>

<hr>

<div style="text-align: justify;">
We will now create and start a buffer. </div>  <br>

<figure style="text-align: center;">
  <img src="./figs/logic/fig7.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 20. Creating and starting buffer inventory. </figcaption>
</figure>

<div style="text-align: justify;">
In the figure, a buffer inventory (<i>doca_buf_inventory</i>) is created and started to manage reusable buffer descriptors required for DMA operations. The inventory is created using <i>doca_buf_inventory_create()</i>, with the size specified by <i>state->base.buf_inventory_size</i>. This inventory will hold <i>doca_buf</i> descriptors that reference portions of the memory mapped in the MMAP. After creation, the buffer inventory is started using <i>doca_buf_inventory_start()</i>, making it ready for buffer allocation and use in DMA tasks. 
</div>

<hr>

<div style="text-align: justify;">
We will now create a PE instance. </div>  <br>

<figure style="text-align: center;">
  <img src="./figs/logic/fig8.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 21. Creating a PE instance. </figcaption>
</figure>

<div style="text-align: justify;">
In the figure, a PE instance is created using <i>doca_pe_create()</i>. The PE is a key component of the DOCA EM, responsible for progressing submitted tasks and invoking their associated callbacks. The PE instance is stored in <i>state->base.pe<i>, which will later be used to drive task progression through polling or event-based mechanisms.
</div>

<hr>

<div style="text-align: justify;">
We will now create and configure the compression context. Add the following to the <i>logic.c</i> file. </div> 

    DOCA_LOG_INFO("Creating Compress");
    EXIT_ON_FAILURE(doca_compress_create(state->base.device, &state->compress));
    state->compress_ctx = doca_compress_as_ctx(state->compress);
    
    EXIT_ON_FAILURE(doca_pe_connect_ctx(state->base.pe, state->compress_ctx));
    
    EXIT_ON_FAILURE(doca_ctx_set_state_changed_cb(state->compress_ctx, compress_state_changed_callback));
    
    doca_error_t result = doca_compress_task_decompress_deflate_set_conf(
        state->compress,
        decompress_deflate_completed_callback,
        decompress_deflate_error_callback,
        1);

<figure style="text-align: center;">
  <img src="./figs/logic/fig9.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 22. Setting up the compression context and configuration. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application sets up the compression context required to perform decompression tasks using the DOCA Compress library. It starts by creating the compression object with <i>doca_compress_create</i> and casting it to a generic context using <i>doca_compress_as_ctx</i>. The context is then connected to the Progress Engine using <i>doca_pe_connect_ctx</i>, allowing asynchronous task processing. A callback is registered with <i>doca_ctx_set_state_changed_cb</i> to monitor changes in the context's state. Finally, the decompression configuration is set using <i>doca_compress_task_decompress_deflate_set_conf</i>, which also registers completion and error callbacks for handling task results.
</div>

<hr>

<div style="text-align: justify;">
We will now set the user data, query buffer size, and start the compression context. Add the following to the <i>logic.c</i> file. </div> 

    union doca_data ctx_user_data = {0};
    ctx_user_data.ptr = state;
    doca_ctx_set_user_data(state->compress_ctx, ctx_user_data);
    
    uint64_t max_buf_size;
    result = doca_compress_cap_task_decompress_deflate_get_max_buf_size(
    doca_dev_as_devinfo(state->base.device),&max_buf_size);
    
    result = doca_ctx_start(state->compress_ctx);

<figure style="text-align: center;">
  <img src="./figs/logic/fig10.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 23. Setting user data and starting the compression context. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application associate’s custom user data with the compression context by assigning the current program state to a <i>doca_data</i> union and setting it using <i>doca_ctx_set_user_data</i>. This allows callback functions to access the program state during task execution. 
</div> <br>

<div style="text-align: justify;">
The code then queries the maximum supported buffer size for deflate decompression using <i>doca_compress_cap_task_decompress_deflate_get_max_buf_size</i>, which helps ensure that allocated buffers remain within hardware limits. Finally, the compression context is started with <i>doca_ctx_start</i>, enabling it to begin processing submitted tasks.
</div>

<hr>

<div style="text-align: justify;">
We will now allocate and map memory for the destination buffer. Add the following to the <i>logic.c</i> file. </div> 

    char *dst_buffer = calloc(1, max_buf_size);
    if (posix_memalign((void **)&dst_buffer, 64, max_buf_size) != 0) {
        DOCA_LOG_ERR(“Failed to allocate aligned memory for destination”);
        Return DOCA_ERROR_NO_MEMORY;
    }
    Memset(dst_buffer, 0, max_buf_size);
    
    result = doca_mmap_set_memrange(state->base.dst_mmap, dst_buffer,     max_buf_size);
    result = doca_mmap_start(state->base.dst_mmap);

<figure style="text-align: center;">
  <img src="./figs/logic/fig11.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 24. Allocating and mapping destination memory for decompression. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application allocates memory for the destination buffer based on the maximum buffer size supported by the decompression operation. It uses <i>posix_memalign</i> to ensure that the allocated memory is aligned to a 64-byte boundary, which is a requirement for DMA compatibility. If the allocation fails, an error is logged, and execution stops. After successful allocation, the memory is initialized using <i>memset</i>. The buffer is then registered with the DOCA memory map using <i>doca_mmap_set_memrange</i>, and the memory map is activated by calling <i>doca_mmap_start</i>. This prepares the buffer for use by the DOCA hardware accelerator.
</div>

<hr>

<div style="text-align: justify;">
We will now load input data and map it to the source memory region. Add the following to the <i>logic.c</i> file. </div> 

    char *file_data = NULL;
    size_t file_size;
    result = read_file("infile", &file_data, &file_size);
    
    result = doca_mmap_set_memrange(state->base.src_mmap, file_data, file_size);
    result = doca_mmap_start(state->base.src_mmap);

<figure style="text-align: center;">
  <img src="./figs/logic/fig12.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 25. Loading and mapping input file data to the source MMAP. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application prepares the input data for decompression by reading it from a file. The function <i>read_file</i> loads the contents of the file named "<i>infile</i>" into memory and stores the data pointer in <i>file_data</i> along with its size in <i>file_size</i>. This buffer is then registered with the source memory map using <i>doca_mmap_set_memrange</i>, allowing the hardware accelerator to access the input data. The source memory map is then activated with <i>doca_mmap_start</i>, completing the preparation of the input buffer for decompression.
</div>

<hr>

<div style="text-align: justify;">
We will now be constructing a DOCA buffer for each address range. Add the following to the <i>logic.c</i> file. </div> 

    struct doca_buf *src_doca_buf;
    result =
        doca_buf_inventory_buf_get_by_addr(state->base.inventory, 
    state->base.src_mmap, file_data, file_size, &src_doca_buf);
    if (result != DOCA_SUCCESS) {
        DOCA_LOG_ERR("Unable to acquire DOCA buffer representing source buffer: %s", doca_error_get_descr(result));
    }
    
    struct doca_buf *dst_doca_buf;
    result = doca_buf_inventory_buf_get_by_addr(state->base.inventory,
                            state->base.dst_mmap,
                            dst_buffer,
                            max_buf_size,
                            &dst_doca_buf);

<figure style="text-align: center;">
  <img src="./figs/logic/fig13.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 26. Creating DOCA buffer descriptors for the source and destination. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application creates DOCA buffer descriptors to represent the source and destination memory regions. It first calls <i>doca_buf_inventory_buf_get_by_addr</i> to wrap the input file data into a <i>src_doca_buf</i>, allowing the DOCA framework to interpret it as a valid buffer within the source memory map. If the operation fails, an error is logged. Next, the destination buffer is wrapped similarly by calling the same function for the destination memory map and storing the result in <i>dst_doca_buf</i>. These descriptors are essential for submitting and tracking tasks within the DOCA pipeline.
</div>

<hr>

<div style="text-align: justify;">
We will now set the data pointer and length for the source buffer. Add the following to the <i>logic.c</i> file. </div> 

    result = doca_buf_set_data(src_doca_buf, file_data, file_size);
    if (result != DOCA_SUCCESS) {
        DOCA_LOG_ERR("Unable to get data length in the DOCA buffer”
                “representing source buffer: %s",
                    doca_error_get_descr(result));
    }

<figure style="text-align: center;">
  <img src="./figs/logic/fig14.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 27. Setting the data pointer and size in the source DOCA buffer. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application assigns the data pointer and size to the source DOCA buffer using the function <i>doca_buf_set_data</i>. This operation is required to let the DOCA runtime know where the actual input data resides and how much of it should be processed. If the call fails, an error is logged indicating that the buffer could not be correctly set up. This step ensures that the buffer is properly configured before being used in a decompression task.
</div>

<hr>

<div style="text-align: justify;">
We will now allocate, initialize, and submit the decompression task. Add the following to the <i>logic.c</i> file. </div> 

    union doca_data task_user_data = {0};
    struct compress_deflate_result task_result = {0};
    task_user_data.ptr = &task_result;
    struct doca_compress_task_decompress_deflate *decompress_task;
    
    result = doca_compress_task_decompress_deflate_alloc_init(state->compress,
        src_doca_buf,
        dst_doca_buf,
        task_user_data,
        &decompress_task);
    
    struct doca_task *task = doca_compress_task_decompress_deflate_as_task(decompress_task);
    state->num_remaining_tasks++;
    
    result = doca_task_submit(task);
    
    state->run_pe_progress = true;

<figure style="text-align: center;">
  <img src="./figs/logic/fig15.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 28. Allocating and submitting a decompression task. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application prepares and submits a decompression task to the DOCA runtime. It begins by creating a user data object and linking it to a result structure that will store the outcome of the decompression, including checksums and status. 
</div> <br>

<div style="text-align: justify;">
The decompression task is then allocated and initialized using <i>doca_compress_task_decompress_deflate_alloc_init</i>, which binds the task to the source and destination buffers and attaches the user data. The task is cast into a generic <i>doca_task</i> and submitted to the PE using <i>doca_task_submit</i>. The application also increments the count of pending tasks and sets a flag to indicate that task processing should begin.
</div>

<hr>

<div style="text-align: justify;">
We will now poll for task completion and prepare the output file. Add the following to the <i>logic.c</i> file. </div> 

    while (state->run_pe_progress) {
        if (doca_pe_progress(state->base.pe) == 0) {
            nanosleep(&ts, &ts);
        }
    }
    
    FILE *out_file = fopen("decompressed", "wr");
    if (out_file == NULL) {
        DOCA_LOG_ERR("Unable to open output file: decompressed");
        return DOCA_ERROR_NO_MEMORY;
    }

<figure style="text-align: center;">
  <img src="./figs/logic/fig16.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 29. Wrapper function that calls run and returns its result. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application enters a polling loop to monitor the progress of submitted tasks using <i>doca_pe_progress</i>. The loop continues as long as the <i>run_pe_progress</i> flag remains true. When no tasks are completed in a polling cycle, the application briefly sleeps using <i>nanosleep</i> to reduce CPU usage. After exiting the loop, it attempts to open a file named "<i>decompressed</i>" in write mode to store the decompressed output. If the file cannot be opened, an error is logged and the function returns an out-of-memory error code, preventing further execution.
</div> 

<hr>

<div style="text-align: justify;">
We will now write the decompressed data to a file and release resources. Add the following to the <i>logic.c</i> file. </div> 

    size_t data_len;
    result = doca_buf_get_data_len(dst_doca_buf, &data_len);
    fwrite(dst_buffer, sizeof(uint8_t), data_len, out_file);
    
    fclose(out_file);
    
    doca_buf_dec_refcount(dst_doca_buf, NULL);
    doca_buf_dec_refcount(src_doca_buf, NULL);
    free(dst_buffer);
    
    return DOCA_SUCCESS;

<figure style="text-align: center;">
  <img src="./figs/logic/fig17.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 30. Wrapper function that calls run and returns its result. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application retrieves the length of the decompressed data stored in the destination DOCA buffer using <i>doca_buf_get_data_len</i>. It then writes the decompressed data to the output file using <i>fwrite</i>. After successfully writing the output, the file is closed to ensure data integrity. The application then releases the reference counts for both the destination and source buffers using <i>doca_buf_dec_refcount</i> and frees the destination buffer memory. This cleanup step ensures that all allocated resources are properly released before the application exits.
</div> 

<hr>

<div style="text-align: justify;">
We define the <i>run_lab_logic</i> function that calls run and returns its status. Add the following to the <i>logic.c</i> file. </div> 

    doca_error_t run_lab_logic(void){
        struct program_state state;
        doca_error_t status = run(&state);
    
        return status;
    }

<figure style="text-align: center;">
  <img src="./figs/logic/fig18.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 31. Definition of the <i>run_lab_logic()</i> function in <i>logic.c</i>. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application defines the <i>run_lab_logic</i> function, which serves as the main entry point to execute the decompression workflow. Inside this function, a <i>program_state</i> structure is declared to hold the application's runtime state. The function then calls <i>run</i>, passing a pointer to this state structure to execute the full decompression logic. Finally, the status returned by <i>run</i> is returned to the caller, indicating whether the operation was completed successfully or encountered an error.
</div> <br>

<div style="text-align: justify;">
Save the changes and close the file.
</div>

<hr>

#### Step 4.4:	Defining the build configuration

<div style="text-align: justify;">
Click on the <i>meson.build</i> file to display the contents of the file. Use the file explorer on the left-hand side of the screen to locate the file. </div> 

<figure style="text-align: center;">
  <img src="./figs/build/fig1.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 32. Inspecting the build file. </figcaption>
</figure>

<hr>

<div style="text-align: justify;">
We see in the figure, <i>meson.build</i> file which has the project metadata, lab name, and enable experimental DOCA APIs predefined. </div> 

<figure style="text-align: center;">
  <img src="./figs/build/fig2.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 33. Project setup, including metadata and experimental API configuration. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, we define the basic configuration for the project using meson.build file. We start by declaring the project name (<i>LAB5</i>) and specifying the programming languages used (<i>C</i> and <i>C++</i>).
</div> <br>

<div style="text-align: justify;">
Next, we retrieve the DOCA SDK version dynamically by reading it from the <i>VERSION</i> file. We also specify that the project uses a proprietary license, set the build type to <i>debug</i> by default, and define the minimum required version of Meson.
</div> <br>

<div style="text-align: justify;">
Then, we assign the name <i>'lab5'</i> to the <i>LAB_NAME</i> variable, which will be reused throughout the build script. Lastly, we add the <i>DOCA_ALLOW_EXPERIMENTAL_API</i> compiler flag to enable experimental DOCA features for both C and C++ source files.
</div>

<hr>

<div style="text-align: justify;">
The required DOCA library dependencies are initialized for the project. </div> 

<figure style="text-align: center;">
  <img src="./figs/build/fig3.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 34. Adding <i>doca-common</i> and <i>doca-dma</i> libraries as dependencies. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the <i>lab5_dependencies</i> list is initialized and populated using the <i>dependency()</i> function to include required DOCA libraries for compilation and linking. The highlighted <i>doca-common</i> and <i>doca-dma</i> libraries are added to <i>lab5_dependencies</i> to enable core utilities and DMA functionality for the lab application.
</div>

<hr>

<div style="text-align: justify;">
The <i>meson.build</i> file has predefined lab source files that are needed for the compilation. </div> 

<figure style="text-align: center;">
  <img src="./figs/build/fig4.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 35. Listing logic and main source files using the lab name prefix. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the list of source files to be compiled for the lab is predefined. A variable named <i>lab_srcs</i> is already set to an array of file names. Each file name is constructed by appending a specific suffix to the <i>LAB_NAME</i> variable (which was previously set to '<i>lab5</i>') to represent different source components. The predefined source list includes '<i>logic.c</i>' and '<i>main.c</i>', ensuring that both the core program logic and the main entry point are compiled into the final executable.
</div>

<hr>

<div style="text-align: justify;">
The include directories required during compilation are predefined in the <i>meson.build</i> file. </div> 

<figure style="text-align: center;">
  <img src="./figs/build/fig5.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 36. Initializing the list of include directories for the build. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the include directories used during the build process are predefined. An empty list named <i>lab_inc_dirs</i> is already initialized, serving as a placeholder for any header file paths that may be added later. Although it's currently empty, this setup ensures the build system is prepared to include necessary directories when required.
</div>

<hr>

<div style="text-align: justify;">
The <i>meson.build</i> file includes the configuration for building the executable using the specified sources and dependencies. </div> 

<figure style="text-align: center;">
  <img src="./figs/build/fig6.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 37. Creating the build target with source files, dependencies, and compiler options. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the configuration for building the final executable is predefined. The <i>executable()</i> function is already used to set the target name by combining the prefix '<i>doca_</i>' with the lab name. The predefined <i>lab_srcs</i> list provides the source files.
</div> <br>

<div style="text-align: justify;">
Additional build settings specify how the project is compiled and linked. The <i>c_args</i> suppresses specific compiler warnings, while dependencies include previously declared DOCA libraries. <i>include_directories</i> sets header paths and <i>install: false</i> prevents system-wide installation.
</div> <br>

<div style="text-align: justify;">
Save the changes and close the file.
</div>

<hr>

#### Step 4.5:	Defining the utils header

<div style="text-align: justify;">
The <i>utils.h</i> file defines utility macros and helper functions that simplify error handling and logging across the program, making the code more readable and consistent. </div> 

<div style="text-align: justify;">
Click on the <i>utils.h</i> file to display the contents of the file. Use the file explorer on the left-hand side of the screen to locate the file. </div> 

<figure style="text-align: center;">
  <img src="./figs/utils_h/fig1.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 38. Inspecting the <i>utils.h</i> file. </figcaption>
</figure>

<hr>

<div style="text-align: justify;">
The <i>utils.h</i> file contains predefined standard and DOCA header file inclusions and a logger for the lab 5 program's callbacks function, as shown inside the gray box. </div> 

<figure style="text-align: center;">
  <img src="./figs/utils_h/fig2.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 39. A header file with predefined single-time header inclusion and header inclusions for defining <i>utils.h</i> file.. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, we begin by checking if the macro <i>UTILS_H</i> is already defined. If it is not, we define it using <i>#define</i>. This ensures the header file is included only once during compilation to avoid duplicate definitions. 
</div> <br>

<div style="text-align: justify;">
The header file includes several essential DOCA headers required for DMA-based memory operations and task progression. <i>doca_log.h</i> provides logging functionality for recording information, warnings, and error messages. doca_mmap.h allows the application to define memory regions that can be mapped and shared between the host and the DPU. <i>doca_buf.h</i> supplies APIs to manage buffer descriptors for accessing and manipulating memory buffers. <i>doca_buf_inventory.h</i> manages a pool of reusable buffer descriptors to optimize memory usage and improve efficiency. <i>doca_ctx.h</i> defines the Context API, which allows the application to configure and control specific DOCA feature contexts, such as DMA. <i>doca_types.h</i> introduces core data types and structures used by other DOCA APIs. <i>doca_dma.h</i> provides DMA APIs for performing asynchronous memory copy operations. Finally, <i>doca_pe.h</i> provides functions for creating and managing PE, which advances task execution and triggers callbacks when tasks complete.
</div> 
<hr>

<div style="text-align: justify;">
We will now see the predefined error-handling macro. </div> 

<figure style="text-align: center;">
  <img src="./figs/utils_h/fig3.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 40. Macro Definition for Handling DOCA Errors and Logging Failures (<i>EXIT_ON_FAILURE</i>). </figcaption>
</figure>

<div style="text-align: justify;">
In the figure, the <i>EXIT_ON_FAILURE</i> macro is defined to simplify error handling across the program. It evaluates a given DOCA API expression and stores its result in _status_. If the result is not DOCA_SUCCESS, an error message is logged using DOCA_LOG_ERR with the function name (<i>__func__</i>) and error description. The macro then immediately returns the error code to propagate the failure. This utility prevents repetitive error-checking code and ensures consistent logging and flow control throughout the application.
At the end of the figure, the #endif directive closes the header guard block started with #ifndef UTILS_H. This ensures that utils.h is included only once per compilation unit, avoiding multiple-inclusion errors and ensuring correct program compilation. 
</div> 

<hr>

<div style="text-align: justify;">
We will now see the predefined <i>read_file</i> function. </div> 

<figure style="text-align: center;">
  <img src="./figs/utils_h/fig4.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 41. Function prototype for reading file contents into memory. </figcaption>
</figure>

<div style="text-align: justify;">
In the figure, the <i>EXIT_ON_FAILURE</i> macro is defined to simplify error handling across the program. It evaluates a given DOCA API expression and stores its result in _status_. If the result is not DOCA_SUCCESS, an error message is logged using DOCA_LOG_ERR with the function name (<i>__func__</i>) and error description. The macro then immediately returns the error code to propagate the failure. This utility prevents repetitive error-checking code and ensures consistent logging and flow control throughout the application.
At the end of the figure, the #endif directive closes the header guard block started with #ifndef UTILS_H. This ensures that utils.h is included only once per compilation unit, avoiding multiple-inclusion errors and ensuring correct program compilation. 
</div> 

<hr>

<div style="text-align: justify;">
We will now see the predefined conditional inclusion guard. </div> 

<figure style="text-align: center;">
  <img src="./figs/utils_h/fig5.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 42. End of header file with conditional inclusion guard. </figcaption>
</figure>

<div style="text-align: justify;">
Close the file.
</div> 

<hr>

#### Step 4.6:	Defining the utils program

<div style="text-align: justify;">
The <i>utils.c</i> file implements utility functions declared in <i>utils.h</i>, such as file reading logic. It handles reading binary input data from disk into memory, which is essential for preparing input buffers used in compression or decompression tasks. </div>

<div style="text-align: justify;">
Click on the <i>utils.c</i> file to display the contents of the file. Use the file explorer on the left-hand side of the screen to locate the file. </div> 

<figure style="text-align: center;">
  <img src="./figs/utils/fig1.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 43. Inspecting the <i>utils.c</i> file. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application begins defining the <i>read_file</i> function, which reads the contents of a file into memory and returns its size. The function takes the file path as input, along with pointers to store the loaded data and its length. This utility function will be responsible for opening the file, reading its contents into a dynamically allocated memory buffer, and returning the number of bytes read. It supports the decompression workflow by supplying the input data from the disk to the program in a format accessible to the DOCA framework. </div> 

<hr>

<div style="text-align: justify;">
We will now open the input file and validate file access. Add the following to <i>utils.c</i> file. </div> 

    FILE *file;
    char *bytes;
    
    file = fopen(path, "rb");
    if (file == NULL)
        return DOCA_ERROR_NOT_FOUND;
    
    if (fseek(file, 0, SEEK_END) != 0) {
        fclose(file);
        return DOCA_ERROR_IO_FAILED;
    }

<figure style="text-align: center;">
  <img src="./figs/utils/fig2.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 44. Opening the input file and validating accessibility. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the <i>read_file</i> function attempts to open the specified file in binary read mode using <i>fopen</i>. It checks whether the file pointer is NULL, which indicates that the file could not be found or opened, and returns a <i>DOCA_ERROR_NOT_FOUND</i> in that case. If the file is successfully opened, the function then seeks to the end of the file using <i>fseek</i> to determine its total size. If this operation fails, it closes the file and returns a <i>DOCA_ERROR_IO_FAILED</i> status, indicating an input or output error during the read process. </div> 

<hr>

<div style="text-align: justify;">
We will now determine the file size and validate its value. Add the following to the <i>utils.c</i> file. </div> 

    long const nb_file_bytes = ftell(file);
    
    if (nb_file_bytes == -1) {
        fclose(file);
        return DOCA_ERROR_IO_FAILED;
    }
    
    if (nb_file_bytes == 0) {
        fclose(file);
        return DOCA_ERROR_INVALID_VALUE;
    }

<figure style="text-align: center;">
  <img src="./figs/utils/fig3.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 45. Getting and validating the size of the input file. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application determines the size of the input file by calling <i>ftell</i> after seeking to the end. This returns the total number of bytes in the file, stored in the variable <i>nb_file_bytes</i>. The function then checks whether <i>ftell</i> returned <i>-1</i>, which indicates an error in determining the file size, and returns an I/O error if so. It also checks if the file size is 0, which would indicate an empty file, and returns an invalid value error. These checks ensure the file is valid and non-empty before proceeding to read its contents. </div> 

<hr>

<div style="text-align: justify;">
We will now allocate aligned memory and reset the file pointer. Add the following to the <i>utils.c</i> file. </div> 

    posix_memalign((void **)&bytes, 64, nb_file_bytes);
    if (bytes == NULL) {
        fclose(file);
        return DOCA_ERROR_NO_MEMORY;
    }
    
    if (fseek(file, 0, SEEK_SET) != 0) {
        free(bytes);
        fclose(file);
        return DOCA_ERROR_IO_FAILED;
    }

<figure style="text-align: center;">
  <img src="./figs/utils/fig4.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 46. Allocating aligned memory and seeking to file start. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application allocates memory to store the contents of the input file using <i>posix_memalign</i>, ensuring 64-byte alignment for optimal hardware access. If the memory allocation fails, the file is closed, and an out-of-memory error is returned. After successful allocation, the file pointer is reset to the beginning of the file using <i>fseek</i> so that the file can be read from the start. If resetting the file pointer fails, the allocated memory is freed, the file is closed, and an I/O error is returned. These steps ensure memory is properly prepared before reading file data into it. </div> 

<hr>

<div style="text-align: justify;">
We will now read file contents into memory and return the result. Add the following to the <i>utils.c</i> file. </div> 

    size_t const read_byte_count = fread(bytes, 1, nb_file_bytes, file);

    fclose(file);
    
    if (read_byte_count != (size_t)nb_file_bytes) {
        free(bytes);
        return DOCA_ERROR_IO_FAILED;
    }
    
    *out_bytes = bytes;
    *out_bytes_len = read_byte_count;
    
    return DOCA_SUCCESS;

<figure style="text-align: center;">
  <img src="./figs/utils/fig5.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 47. Reading file data into memory and finalizing output. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application reads the contents of the input file into the previously allocated aligned memory using <i>fread</i>. The number of bytes successfully read is stored in <i>read_byte_count</i>. After reading, the file is closed to release the file handle. The function checks if the number of bytes read matches the expected file size. If not, it frees the memory and returns an I/O error. Otherwise, it sets the output pointers to return the read data and its size, and returns a success code, indicating the file was loaded into memory correctly. </div> <br>

<div style="text-align: justify;">
Save the changes and close the file.
</div>

<hr>

#### Step 4.7:	Defining the callbacks header

<div style="text-align: justify;">
The <i>callbacks.h</i> file stores declarations of callback functions and helper functions used for processing DMA Memcpy task completions and errors, allowing these functions to be shared across multiple source files (like logic.c in the lab5 directory). </div>


<div style="text-align: justify;">
Click on the <i>callbacks.h</i> file to display the contents of the file. Use the file explorer on the left-hand side of the screen to locate the file. </div> 

<figure style="text-align: center;">
  <img src="./figs/callbacks_h/fig1.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 48. Inspecting the <i>callbacks_h</i> file. </figcaption>
</figure>

<hr>

<div style="text-align: justify;">
The <i>callbacks.h</i> file contains predefined standard and DOCA header file inclusions for the lab 5 program's callbacks function, as shown inside the gray box. </div> 

<figure style="text-align: center;">
  <img src="./figs/callbacks_h/fig2.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 49. Inspecting the headers. </figcaption>
</figure>

<div style="text-align: justify;">
In the figure, the standard C library headers included are <i>stdlib.h</i>, which provides memory management, process control, and utility functions, and <i>stdint.h</i>, which defines fixed-width integer types for portability. Among the DOCA core utility headers, <i>doca_mmap.h</i> provides functions for creating and managing memory mappings used to share memory between host and DOCA devices. <i>doca_buf.h</i> defines APIs for managing buffer descriptors that represent memory regions. <i>doca_buf_inventory.h</i> enables efficient pooling and reuse of buffer descriptors. <i>doca_ctx.h</i> defines context objects used to encapsulate the configuration and state of specific DOCA features. <i>doca_dma.h</i> exposes APIs for performing DMA-based memory copy operations. <i>doca_types.h</i> introduces common types and data structures used across DOCA libraries. <i>doca_log.h</i> enables message logging, useful for debugging and monitoring. Finally, <i>doca_pe.h</i> provides APIs to manage the DOCA PE, which drives the execution of submitted tasks.
</div> <br>

<div style="text-align: justify;">
In addition, two user-defined headers are included: <i>definitions.h</i>, which defines shared data structures used across the program, and <i>utils.h</i>, which provides utility macros and helper functions used in task management and buffer operations.
</div>

<hr>

<div style="text-align: justify;">
We will now declare callback functions for DMA task completion and error handling.  </div> 

<figure style="text-align: center;">
  <img src="./figs/callbacks_h/fig3.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 50. Declaring DMA task completion and error callback functions. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application declares two callback functions for handling the outcome of DMA Memcpy tasks. The <i>dma_memcpy_completed_callback</i> is invoked when a DMA task finishes successfully and provides access to task-specific and context-specific data through the <i>task_user_data</i> and <i>ctx_user_data</i> parameters. Similarly, the <i>dma_memcpy_error_callback</i> is triggered when a DMA task fails. These callbacks are part of the asynchronous task management model and are essential for responding to task completion events or errors during runtime.
</div>

<hr>

<div style="text-align: justify;">
We will now declare callbacks for compression state changes and task completion.  </div> 

<figure style="text-align: center;">
  <img src="./figs/callbacks_h/fig4.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 51. Declaring compression context and task completion callbacks. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application declares three callback functions used during the compression and decompression workflow. The <i>compress_state_changed_callback</i> is triggered when the state of the compression context changes, providing insight into state transitions via the previous and next state parameters. The <i>decompress_deflate_completed_callback</i> is called when a decompression task finishes successfully, allowing the program to handle the output and proceed accordingly. These callbacks are essential for maintaining control over asynchronous operations and ensuring appropriate actions are taken when specific events occur in the compression context.
</div> 

<hr>

<div style="text-align: justify;">
We will now declare a callback to handle decompression task errors.  </div> 

<figure style="text-align: center;">
  <img src="./figs/callbacks_h/fig5.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 52. Declaring a callback for decompression task failure. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application declares the <i>decompress_deflate_error_callback</i> function, which is called whenever a decompression task fails. This callback receives a pointer to the failed task along with associated user and context data. It enables the application to log errors, perform diagnostics, and clean up resources in response to task failure. Including this callback ensures that error handling is integrated into the asynchronous decompression workflow.
</div> <br>

<div style="text-align: justify;">
Close the file
</div>

<hr>

#### Step 4.8:	Defining the callbacks program


<div style="text-align: justify;">
The <i>callbacks.c</i> file implements the callback functions and helper routines used to handle DMA Memcpy task completions, verify data correctness, and manage task-related resource cleanup. </div>

<figure style="text-align: center;">
  <img src="./figs/callbacks/fig1.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 53. Inspecting the callbacks program file. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application begins defining two key callback functions used in the compression workflow. The <i>compress_state_changed_callback</i> is triggered when the state of the compression context transitions from one phase to another, such as from idle to running. This is useful for debugging or updating program flow based on context status. The <i>decompress_deflate_completed_callback</i> is invoked when a decompression task successfully finishes, providing access to task-specific data and context state for further processing or verification. These callbacks are critical for handling asynchronous state and task updates in DOCA applications.
</div> <br>

<div style="text-align: justify;">
The application defines the <i>decompress_deflate_error_callback</i> function, which is executed when a decompression task fails. It receives the failed task descriptor along with user and context data, enabling the program to log the error, clean up resources, or trigger recovery mechanisms. This function complements the successful completion callback and ensures that the application remains robust by gracefully handling task-level failures.
</div>

<hr>

<div style="text-align: justify;">
We will now extract program state from user data in the compression state callback. Add the following to the <i>callbacks.c</i> file. </div>

    (void)ctx;
    (void)prev_state;
    
    struct program_state *state = (struct program_state *)user_data.ptr;

<figure style="text-align: center;">
  <img src="./figs/callbacks/fig2.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 54. Extracting program state from user data in the compression callback. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the application extracts the program state structure from the <i>user_data</i> parameter passed into the compression context state change callback. The <i>ctx</i> and <i>prev_state</i> parameters are explicitly marked as unused to avoid compiler warnings. The <i>user_data.ptr</i> field, which was previously set during context initialization, is cast back to a <i>program_state</i> pointer. This allows the callback to access and update the application state when reacting to changes in the compression context’s status.
</div> 

<hr>

<div style="text-align: justify;">
We will now handle different compression context states using a switch-case block. Add the following to the <i>callbacks.c</i> file. </div>

    switch (next_state) {
        case DOCA_CTX_STATE_IDLE:
            DOCA_LOG_INFO("Compress context has been stopped");
            state->run_pe_progress = false;
            break;
        case DOCA_CTX_STATE_STARTING:
            DOCA_LOG_ERR("Compress context entered into starting"
                "state. Unexpected transition");
            break;
        case DOCA_CTX_STATE_RUNNING:
            DOCA_LOG_INFO("Compress context is running");
            break;
        case DOCA_CTX_STATE_STOPPING:
            DOCA_LOG_INFO("Compress context entered into stopping state." 
                "Any inflight tasks will be flushed");
            break;
        default:
            break;
    }

<figure style="text-align: center;">
  <img src="./figs/callbacks/fig3.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 55. Handling compression context transitions. </figcaption>
</figure>

<div style="text-align: justify;">
This figure shows the handling of state transitions in the compression context using a switch-case statement on <i>next_state</i>. The callback logs appropriate messages and updates internal flags depending on whether the compression context is idle, starting, running, or stopping. For instance, when the context enters the <i>IDLE state</i>, it logs that the compression has stopped and disables progress tracking. Warnings are raised for unexpected states, such as <i>STARTING</i>, and each state provides information about the program's operation stage.
</div> 

<hr>

<div style="text-align: justify;">
We will now handle decompression task completion and finalize results. Add the following to the <i>callbacks.c</i> file. </div>

    struct program_state *resources = (
        struct program_state *)ctx_user_data.ptr;
    struct compress_deflate_result *result = (
        struct compress_deflate_result *)task_user_data.ptr;
    
    DOCA_LOG_INFO("Decompress task was done successfully");
    
    result->crc_cs = doca_compress_task_decompress_deflate_get_crc_cs(decompress_task);
    result->adler_cs = doca_compress_task_decompress_deflate_get_adler_cs(decompress_task);
    result->status = DOCA_SUCCESS;
    
    doca_task_free(doca_compress_task_decompress_deflate_as_task(decompress_task));
    --resources->num_remaining_tasks;
    if (resources->num_remaining_tasks == 0)
        (void)doca_ctx_stop(resources->compress_ctx);

<figure style="text-align: center;">
  <img src="./figs/callbacks/fig4.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 56. Finalization steps after the decompression task completion. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the <i>decompress_deflate_completed_callback</i> function is shown, which executes after a decompression task finishes successfully. It extracts the program state and result structures from user data, logs the successful event, retrieves CRC and Adler checksums, and sets the result status. The task is then freed, the remaining task counter is decremented, and if no tasks remain, the compression context is stopped to conclude processing.
</div> 

<hr>

<div style="text-align: justify;">
The <i>callbacks.c</i> file declares handling DMA task errors, shown in the figure with a gray box.</div>

    struct program_state *resources = (
    struct program_state *)ctx_user_data.ptr;
    struct doca_task *task = doca_compress_task_decompress_deflate_as_task(decompress_task);
    struct compress_deflate_result *result = (
    struct compress_deflate_result *)task_user_data.ptr;
    
    result->status = doca_task_get_status(task);
    DOCA_LOG_ERR("Decompress task failed: %s", 
    doca_error_get_descr(result->status));
    doca_task_free(task);
    --resources->num_remaining_tasks;
    if (resources->num_remaining_tasks == 0)
        (void)doca_ctx_stop(resources->compress_ctx);

<figure style="text-align: center;">
  <img src="./figs/callbacks/fig5.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 57. Error handling and cleanup after failed decompression task. </figcaption>
</figure>

<div style="text-align: justify;">
In this figure, the <i>decompress_deflate_error_callback</i> function handles the logic when a decompression task fails. It retrieves the program state, task, and result structures from user-provided data. The status of the failed task is fetched and logged as an error. Following this, the task is freed, the number of remaining tasks is decremented, and the compression context is stopped if no tasks are left.
</div> <br>

<div style="text-align: justify;">
Save the changes and close the file.
</div>

<hr>

#### Step 4.9:	Inspecting the output files


<div style="text-align: justify;">
Click on the <i>decompressed</i> file to display the contents of the file. Use the file explorer on the left-hand side of the screen to locate the file.</div>

<figure style="text-align: center;">
  <img src="./figs/output/fig1.png" width="600" style="display: block; margin: 0 auto;">
  <figcaption>Figure 58. Inspecting the decompressed file. </figcaption>
</figure>

<div style="text-align: justify;">
In this lab, the infile refers to the compressed data file that serves as the input for the decompression operation.
</div>

- This file must already exist and contain valid compressed data.
- It is passed to the program so that the contents can be decompressed and restored.

<div style="text-align: justify;">
The decompressed file is the destination output file where the decompressed data will be written.
</div>

- This file will be created or overwritten during the decompression process.
- It contains the original uncompressed data restored from the input file.

<div style="text-align: justify;">
In this figure, we observe the content of the decompressed output file. The file displays the original text "<i>some text</i>" that was present before compression. This confirms that the decompression operation has successfully restored the original data. </div> <br>

<div style="text-align: justify;">
Close the <i>decompressed</i> file.
</div>

<hr>

### Step 5: Enabling internet on BlueField SmartNIC

<div style="text-align: justify;">
This set of commands below configures the system to function as a network gateway using Network Address Translation. It establishes packet forwarding between the internal interface <i>tmfifo_net0</i> and the external interface <i>enp3s0</i>, allowing devices on the internal network to communicate with external networks through the host. NAT masquerading is applied to outgoing traffic so that packets appear to originate from the host’s external IP address. Additionally, IP forwarding is enabled in the kernel to permit routing of packets between interfaces. It enables the host to act as a bridge that provides internet connectivity to internal network devices.
</div>

In [30]:
stdout, stderr = node1.execute(f'sudo iptables -t nat -A POSTROUTING -o enp3s0 -j MASQUERADE', quiet = True)
stdout, stderr = node1.execute(f'sudo iptables -A FORWARD -i enp3s0 -o tmfifo_net0 -m state --state RELATED,ESTABLISHED -j ACCEPT', quiet = True)
stdout, stderr = node1.execute(f'sudo iptables -A FORWARD -i tmfifo_net0 -o enp3s0 -j ACCEPT', quiet = True)
stdout, stderr = node1.execute(f'sudo sysctl -w net.ipv4.ip_forward=1', quiet = True)
stdout, stderr = node1.execute(
    "ssh ubuntu@192.168.100.2 \"echo -e 'nameserver 8.8.8.8\nnameserver 192.168.100.1' | sudo tee /etc/resolv.conf > /dev/null\""
)

### Step 6:	Building the DOCA application


Create a directory for the files to be uploaded

In [13]:
stdout, stderr = node1.execute(
    f'mkdir DOCA_Lab'
)

To upload the DOCA code to Node1, issue the following commands below.

In [14]:
node1.upload_file('lab_files/main.c', '/home/ubuntu/DOCA_Lab/main.c')
node1.upload_file('lab_files/callbacks.c', '/home/ubuntu/DOCA_Lab/callbacks.c')
node1.upload_file('lab_files/callbacks.h', '/home/ubuntu/DOCA_Lab/callbacks.h')
node1.upload_file('lab_files/utils.c', '/home/ubuntu/DOCA_Lab/utils.c')
node1.upload_file('lab_files/utils.h', '/home/ubuntu/DOCA_Lab/utils.h')
node1.upload_file('lab_files/decompressed', '/home/ubuntu/DOCA_Lab/decompressed')
node1.upload_file('lab_files/infile', '/home/ubuntu/DOCA_Lab/infile')
node1.upload_file('lab_files/logic.c', '/home/ubuntu/DOCA_Lab/logic.c')
node1.upload_file('lab_files/meson.build', '/home/ubuntu/DOCA_Lab/meson.build')
node1.upload_file('lab_files/definitions.h', '/home/ubuntu/DOCA_Lab/definitions.h')

<SFTPAttributes: [ size=1193 uid=1000 gid=1000 mode=0o100664 atime=1760332684 mtime=1760332684 ]>

We send the files to Bluefield by first zipping the program folder.

In [15]:
stdout, stderr = node1.execute(
    f'tar -czvf DOCA_Lab.tar.gz DOCA_Lab/'
)

DOCA_Lab/
DOCA_Lab/callbacks.h
DOCA_Lab/decompressed
DOCA_Lab/main.c
DOCA_Lab/infile
DOCA_Lab/definitions.h
DOCA_Lab/logic.c
DOCA_Lab/utils.c
DOCA_Lab/utils.h
DOCA_Lab/callbacks.c
DOCA_Lab/meson.build


We copy the zip file from the host to BlueField as shown by the commands below.

In [24]:
stdout, stderr = node1.execute(
    "scp /home/ubuntu/DOCA_Lab.tar.gz ubuntu@192.168.100.2:/home/ubuntu"
)

Verifying if the copy was successful.

In [25]:
stdout, stderr = node1.execute(
    "ssh ubuntu@192.168.100.2 'ls'"
)

DOCA_Lab.tar.gz


Unzip the program files in the BlueField.

In [26]:
stdout, stderr = node1.execute(
    "ssh ubuntu@192.168.100.2 'tar -xzvf DOCA_Lab.tar.gz'"
)

DOCA_Lab/
DOCA_Lab/callbacks.h
DOCA_Lab/decompressed
DOCA_Lab/main.c
DOCA_Lab/infile
DOCA_Lab/definitions.h
DOCA_Lab/logic.c
DOCA_Lab/utils.c
DOCA_Lab/utils.h
DOCA_Lab/callbacks.c
DOCA_Lab/meson.build


Build the program code by using meson. 

In [35]:
stdout, stderr = node1.execute(
    "ssh ubuntu@192.168.100.2 'cd ~/DOCA_Lab && meson build'"
)

The Meson build system
Version: 0.61.2
Source dir: /home/ubuntu/DOCA_Lab
Build dir: /home/ubuntu/DOCA_Lab/build
Build type: native build
Program cat found: YES (/usr/bin/cat)
Project name: LAB5
Project version: 3.0.0058
C compiler for the host machine: cc (gcc 11.4.0 "cc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0")
C linker for the host machine: cc ld.bfd 2.38
C++ compiler for the host machine: c++ (gcc 11.4.0 "c++ (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0")
C++ linker for the host machine: c++ ld.bfd 2.38
Host machine cpu family: aarch64
Host machine cpu: aarch64
Found pkg-config: /usr/bin/pkg-config (0.29.2)
Run-time dependency doca-common found: YES 3.0.0058
Run-time dependency doca-dma found: YES 3.0.0058
Run-time dependency doca-compress found: YES 3.0.0058
Build targets in project: 1

Found ninja-1.10.1 at /usr/bin/ninja


<div style="text-align: justify;">
In this figure, the command meson build initializes the build system in lab5 directory. It sets up the build directory, detects compilers (gcc and g++ version 13.3.0), verifies required dependencies like doca-common and doca-dma, and confirms the presence of tools like pkg-config and ninja. The project is configured successfully with version 3.0.0058. </div>

<div style="text-align: justify;">
Initialize the build environment for the project. Write the following command in the terminal. </div>

In [36]:
stdout, stderr = node1.execute(
    "ssh ubuntu@192.168.100.2 'cd ~/DOCA_Lab && ninja -C build'"
)

ninja: Entering directory `build'
[1/5] Compiling C object doca_lab5.p/utils.c.o
[2/5] Compiling C object doca_lab5.p/main.c.o
[3/5] Compiling C object doca_lab5.p/callbacks.c.o
[4/5] Compiling C object doca_lab5.p/logic.c.o
[5/5] Linking target doca_lab5


<div style="text-align: justify;">
In this figure, we execute the command <i>ninja -C build</i>, which tells Ninja to enter the build directory and start the compilation process based on the configuration generated by Meson. Ninja then compiles the source files and performs the linking step to generate the final executable. The message [5/5] Linking target <i>doca_lab5</i> indicates that all build steps were completed successfully, and the binary <i>doca_lab5</i> has been created.
</div>

In [37]:
stdout, stderr = node1.execute(
    "ssh ubuntu@192.168.100.2 'cd ~/DOCA_Lab && ./build/doca_lab5'"
)

[05:32:03:704123][202532][DOCA][INF][main.c:33][main] Starting the program LAB 5
[05:32:03:704224][202532][DOCA][INF][logic.c:32][run] Initializing program state
[05:32:03:704228][202532][DOCA][INF][logic.c:35][run] Allocating buffer with size of 1024
[05:32:03:710649][202532][DOCA][INF][logic.c:64][run] Creating MMAP
[05:32:03:710672][202532][DOCA][INF][logic.c:70][run] Creating buf inventory
[05:32:03:710678][202532][DOCA][INF][logic.c:75][run] Creating PE
[05:32:03:711046][202532][DOCA][INF][logic.c:78][run] Creating Compress
[05:32:03:933520][202532][DOCA][INF][callbacks.c:27][compress_state_changed_callback] Compress context is running
[05:32:03:935722][202532][DOCA][INF][callbacks.c:48][decompress_deflate_completed_callback] Decompress task was done successfully
[05:32:03:935741][202532][DOCA][INF][callbacks.c:30][compress_state_changed_callback] Compress context entered into stopping state.Any inflight tasks will be flushed
[05:32:04:051833][202532][DOCA][INF][callbacks.c:19][co

In this figure, we run the application using the command *./build/doca_lab5*. The terminal output shows a sequence of log messages:

- The program starts successfully (Starting the program LAB 5).
- It initializes the program state.
- It allocates a buffer of size 32768 bytes in memory.
- It creates a DOCA MMAP object to map the memory buffer.
- It creates a DOCA buffer inventory.
- It creates the PE.
- It creates and configures the DMA context.
- The DMA context is started.
- It allocates DMA Memcpy tasks.
- It submits the DMA Memcpy tasks to the PE.
- The application polls the PE until all tasks are completed. 
- The program logs "All tasks are completed", confirming that all DMA Memcpy tasks finished successfully.
- The program logs that LAB 5 finished successfully, confirming that the application completed its run without errors.


### Step 7: Delete the slice

This concludes the Lab. Please delete your slice when you are done with your experiment.

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
fablib = fablib_manager()
slice = fablib.get_slice(name="DOCA_LAB")
slice.delete()

### References

1.	NVIDIA Developer, ‘’DOCA Documentation’’, [Online]. Available: https://docs.nvidia.com/doca/sdk/doca+core/index.html.